# County Rainfall and Planting Advisory

**ADEN AHMED MOHAMED — Cohort 1M DEVIS**

Analyse the supplied monthly rainfall for five counties using NumPy and export a county summary. Values are in millimetres; columns run from January to December.

In [1]:
import numpy as np
import csv

counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]
clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],   # Nairobi
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],    # Nakuru
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],   # Kisumu
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70],   # Mombasa
]
machakos_raw = ["48", "35", "88", "N/A", "140", " 20 ",
                "9", "12", "", "45", "182", "96"]
months = np.array(["January", "February", "March", "April", "May", "June",
                   "July", "August", "September", "October", "November", "December"])

## Task 1: Clean Machakos readings

Convert each raw string with int(); surrounding whitespace is accepted. Catch ValueError specifically, calculate the rounded mean of the valid values, and fill only invalid positions.

In [2]:
def clean_machakos(raw_values, month_names):
    """Return integer readings with invalid strings replaced by the rounded valid mean."""
    values = []
    invalid_indices = []
    for index, value in enumerate(raw_values):
        try:
            values.append(int(value))
        except ValueError:
            values.append(None)
            invalid_indices.append(index)
    valid_values = [value for value in values if value is not None]
    if not valid_values:
        raise ValueError("At least one valid reading is needed for mean imputation.")
    replacement = round(sum(valid_values) / len(valid_values))
    print(f"Valid Machakos mean: {sum(valid_values) / len(valid_values):.2f} mm")
    for index in invalid_indices:
        values[index] = replacement
        print(f"Imputed {month_names[index]}: {replacement} mm")
    return values

machakos_clean = clean_machakos(machakos_raw, months)
print("Clean Machakos row:", machakos_clean)

Valid Machakos mean: 67.50 mm
Imputed April: 68 mm
Imputed September: 68 mm
Clean Machakos row: [48, 35, 88, 68, 140, 20, 9, 12, 68, 45, 182, 96]


## Task 2: Build the rainfall array

Combine the four supplied rows and the cleaned Machakos row into one integer NumPy array. Rows represent counties and columns represent months.

In [3]:
R = np.array(clean_rows + [machakos_clean], dtype=np.int64)
print(R)
print("Shape:", R.shape)
print("Dtype:", R.dtype)

[[ 64  56  92 219 176  35  18  24  31  61 165 104]
 [ 35  42  70 145 120  78  96 105  72  55  60  38]
 [ 58  90 150 240 180  95  70  85  90  88 130  95]
 [ 30  15  55 150 240 110  90  65  60  90 100  70]
 [ 48  35  88  68 140  20   9  12  68  45 182  96]]
Shape: (5, 12)
Dtype: int64


## Task 3: Annual totals and monthly averages

Sum across columns (axis=1) for each county’s annual total. Average down rows (axis=0) for each month’s mean across the five counties. Loops below only format output.

In [4]:
annual_totals = R.sum(axis=1)
monthly_averages = R.mean(axis=0)
print("Annual rainfall totals")
for county, total in zip(counties, annual_totals):
    print(f"{county:<10} {total:>7,} mm")
print("\nMonthly averages across all counties")
for month, average in zip(months, monthly_averages):
    print(f"{month:<10} {average:>7.2f} mm")

Annual rainfall totals
Nairobi      1,045 mm
Nakuru         916 mm
Kisumu       1,371 mm
Mombasa      1,075 mm
Machakos       811 mm

Monthly averages across all counties
January      47.00 mm
February     47.60 mm
March        91.00 mm
April       164.40 mm
May         171.20 mm
June         67.60 mm
July         56.60 mm
August       58.20 mm
September    64.20 mm
October      67.80 mm
November    127.40 mm
December     80.60 mm


## Task 4: Wettest month per county

Use np.argmax(axis=1) and index the month-name array. Pair the counties and results with zip() for printing. If values tie, argmax selects the earliest month.

In [5]:
wettest_indices = np.argmax(R, axis=1)
wettest_months = months[wettest_indices]
wettest_amounts = R[np.arange(R.shape[0]), wettest_indices]
for county, month, amount in zip(counties, wettest_months, wettest_amounts):
    print(f"{county:<10} {month:<10} {amount:>4} mm")

Nairobi    April       219 mm
Nakuru     April       145 mm
Kisumu     April       240 mm
Mombasa    May         240 mm
Machakos   November    182 mm


## Task 5: Count dry months

A dry month has rainfall strictly below 50 mm. Sum the boolean mask across each row to count dry months without a calculation loop.

In [6]:
dry_mask = R < 50
dry_counts = dry_mask.sum(axis=1)
for county, count in zip(counties, dry_counts):
    print(f"{county:<10} {count} dry months")

Nairobi    4 dry months
Nakuru     3 dry months
Kisumu     0 dry months
Mombasa    2 dry months
Machakos   6 dry months


## Task 6: Recommended planting windows

Add shifted column slices to calculate all eleven adjacent two-month totals simultaneously, then use argmax for each county. This follows the assignment’s rainfall rule, excludes December–January wrapping, and selects the earliest window if tied.

In [7]:
two_month_totals = R[:, :-1] + R[:, 1:]
window_indices = np.argmax(two_month_totals, axis=1)
planting_windows = np.char.add(
    np.char.add(months[window_indices], "–"), months[window_indices + 1]
)
window_totals = two_month_totals[np.arange(R.shape[0]), window_indices]
for county, window, total in zip(counties, planting_windows, window_totals):
    print(f"{county:<10} {window:<20} {total:>4} mm")

Nairobi    April–May             395 mm
Nakuru     April–May             265 mm
Kisumu     April–May             420 mm
Mombasa    April–May             390 mm
Machakos   November–December     278 mm


## Task 7: Export the CSV

Write one header and exactly five county records with the standard-library csv module. The export loop serialises the vectorised results; it does not calculate them.

In [8]:
with open("rainfall_summary.csv", "w", newline="", encoding="utf-8") as output_file:
    writer = csv.writer(output_file)
    writer.writerow(["county", "annual_total_mm", "wettest_month",
                     "dry_months", "planting_window"])
    writer.writerows(zip(counties, annual_totals.tolist(), wettest_months.tolist(),
                         dry_counts.tolist(), planting_windows.tolist()))
print("Saved rainfall_summary.csv: 5 county records.")

Saved rainfall_summary.csv: 5 county records.


## Task 8: Advisory and data quality

Translate the computed patterns into practical advice while distinguishing this assignment’s rainfall-based window from a verified agronomic forecast.

## Planting Advisory

Use April–May as the rainfall-based planning window for Nairobi, Nakuru, Kisumu and Mombasa, with combined totals of 395, 265, 420 and 390 mm respectively. For Machakos, the supplied method selects November–December, with 278 mm, so the NGO should advise farmers to prepare inputs before November while checking local rainfall onset. Prioritise water-storage and moisture-conservation advice for Machakos, which has 6 dry months below 50 mm, and Nairobi, which has 4. Kisumu has no dry months under this threshold, but farmers should still monitor drainage and avoid assuming rainfall is evenly distributed within each month. Machakos’s April and September readings were imputed at 68 mm each, so its annual total and seasonal comparisons are estimates that should be checked against station records. These monthly observations and the highest two-month rainfall rule are planning aids; crop choice, soil conditions and current local forecasts should guide actual planting decisions.

## Data Quality Notes

- The ten valid Machakos values sum to 675 mm, giving a mean of 67.5 mm; Python round() yields an imputation of 68 mm.
- April (N/A) and September (empty string) were invalid; June’s whitespace-padded value was successfully converted to 20 mm.
- Imputation treats missing months as having the same estimated value and may hide genuine seasonal variation. Machakos’s annual total, April/September cross-county averages, dry-month count and window comparison therefore depend on these estimates.
- No year, station coverage, daily distribution or multi-year history was supplied, so these data do not establish climate normals or a current forecast.
- Dry means strictly below 50 mm. Adjacent windows run January–February through November–December; ties resolve to the earliest position.
- NumPy performs all rainfall aggregation, masking and argmax calculations. Python loops are limited to parsing/error reporting, readable output and CSV serialisation.